In [0]:
from config.f1_config import add_ingestion_metadata,CATALOG, BRONZE_SCHEMA, SILVER_SCHEMA
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType,TimestampType, FloatType
import pyspark.sql.functions as F

SILVER_TABLE = "f1_workspace.silver.drivers"

In [0]:
df_bronze_drivers = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.drivers")

TRASFORM AND FLATTEN DATA


In [0]:
def transform_circuits_silver(df_bronze):
    
    return (
        df_bronze.select(
            F.trim(F.col("driverId")).alias("Driver_Id"),
            F.trim(F.initcap(F.col("code"))).alias("Driver_Code"),
            F.trim(F.initcap(F.col("givenName"))).alias("First_Name"),
            F.trim(F.initcap(F.col("familyName"))).alias("Last_Name"),
            F.trim(F.col("dateOfBirth")).cast("date").alias("Date_Of_Birth"),
            F.trim(F.initcap(F.col("nationality"))).alias("Nationality"),
            F.trim(F.col("permanentNumber")).cast("int").alias("Driver_Number"),   
        )
        .withColumn("silver_processed_timestamp", F.current_timestamp())
    )


df_silver_circuits = transform_circuits_silver(df_bronze_drivers)


In [0]:
df_silver_circuits.write.format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", True)\
    .saveAsTable(SILVER_TABLE)